# 04 - Experiments on top of TF-IDF

| Experiment | Change vs reference | Why (EDA) |
|---|---|---|
| `reference` | best config from 02 | fixed point of comparison |
| `class_weight` | `class_weight="balanced"` | ai 44.8% vs human 24.5%, metric is macro-F1 |
| `domain` | + one-hot `domain` | class share depends strongly on domain (arxiv 84% ai) |
| `style` | + scaled stylometric features | ai: longer words, more even sentences |
| `all` | all three | combined effect |

- Same train subsample and validation split as 02; every transform is fitted on the train part only
- Kaggle: CPU is enough, Internet on, competition data attached

## Setup

In [ ]:
import subprocess
import sys

REPO = "https://github.com/mohinska/ai_hw3_AIGenTextDetection.git"
REPO_DIR = "/tmp/repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import common
from common import LABELS, TEXT_COL, LABEL_COL, SEED
from features import fit_tfidf, style_features, tfidf_transform

warnings.filterwarnings("ignore", category=UserWarning)

## Config

In [ ]:
N_TRAIN = 60_000                           # same subsample as 02
REF_FEATURES = ["char_wb_2-5"]             # best feature set from 02
REF_C = 2.0                                 # best C from 02
DROP_STYLE = ["newlines_per_100w"]          # no signal in EDA (median 0 in every class)

EXPERIMENTS = {
    "reference":    dict(class_weight=None,       domain=False, style=False),
    "class_weight": dict(class_weight="balanced", domain=False, style=False),
    "domain":       dict(class_weight=None,       domain=True,  style=False),
    "style":        dict(class_weight=None,       domain=False, style=True),
    "all":          dict(class_weight="balanced", domain=True,  style=True),
}
SUBMISSION_PATH = os.path.join(common.WORK_DIR, "submission_tfidf_best.csv")

## Data

In [ ]:
train_df, test_df, sample_sub = common.load_data()
trn, val = common.split_train_val(train_df)
trn = common.subsample(trn, N_TRAIN)
y_trn, y_val = trn[LABEL_COL].to_numpy(), val[LABEL_COL].to_numpy()
parts = {"trn": trn, "val": val, "test": test_df}
print({k: len(v) for k, v in parts.items()})

## Feature blocks

| Block | Fitted on |
|---|---|
| TF-IDF (`REF_FEATURES`) | train part |
| one-hot `domain` | train part |
| style features, standardised | train part |

In [ ]:
t0 = time.time()
vecs = fit_tfidf(REF_FEATURES, trn[TEXT_COL].fillna(""))
X_tfidf = {k: tfidf_transform(vecs, df[TEXT_COL].fillna("")) for k, df in parts.items()}

ohe = OneHotEncoder(handle_unknown="ignore").fit(trn[["domain"]])
X_domain = {k: ohe.transform(df[["domain"]]).tocsr() for k, df in parts.items()}
print(f"tfidf + domain: {(time.time() - t0) / 60:.1f} min, {X_tfidf['trn'].shape[1]:,} tfidf features")

In [ ]:
t0 = time.time()
style_raw = {k: pd.DataFrame([style_features(t) for t in df[TEXT_COL].fillna("")]).drop(columns=DROP_STYLE)
             for k, df in parts.items()}
scaler = StandardScaler().fit(style_raw["trn"])
X_style = {k: csr_matrix(scaler.transform(v)) for k, v in style_raw.items()}
print(f"style: {(time.time() - t0) / 60:.1f} min, features: {list(style_raw['trn'].columns)}")

## Run experiments

In [ ]:
def build_X(part, cfg):
    blocks = [X_tfidf[part]]
    if cfg["domain"]:
        blocks.append(X_domain[part])
    if cfg["style"]:
        blocks.append(X_style[part])
    return hstack(blocks).tocsr()


rows, models = [], {}
for name, cfg in EXPERIMENTS.items():
    t0 = time.time()
    clf = LogisticRegression(C=REF_C, class_weight=cfg["class_weight"], max_iter=2000, random_state=SEED)
    clf.fit(build_X("trn", cfg), y_trn)
    proba = clf.predict_proba(build_X("val", cfg))
    pred = clf.classes_[proba.argmax(1)]
    common.save_proba(val["id"], proba, clf.classes_, f"tfidf_{name}", "val")
    per_class = f1_score(y_val, pred, labels=LABELS, average=None)
    rows.append({"experiment": name, "macro_f1": f1_score(y_val, pred, average="macro"),
                 **{f"f1_{l}": f for l, f in zip(LABELS, per_class)}, "fit_min": (time.time() - t0) / 60})
    models[name] = (clf, pred)

results = pd.DataFrame(rows)
results["delta_vs_ref"] = results["macro_f1"] - results.loc[results.experiment == "reference", "macro_f1"].item()
results = results.round(4)
results.to_csv(os.path.join(common.WORK_DIR, "outputs", "tfidf_experiments.csv"), index=False)
results

In [ ]:
# macro-F1 per domain: reference vs best experiment
BEST = results.sort_values("macro_f1", ascending=False).experiment.iloc[0]
pd.concat({n: common.f1_by(y_val, models[n][1], val["domain"])["macro_f1"] for n in ["reference", BEST]},
          axis=1).assign(n=val["domain"].value_counts()).sort_values("n", ascending=False)

## Test predictions for the best experiment

In [ ]:
clf = models[BEST][0]
test_proba = clf.predict_proba(build_X("test", EXPERIMENTS[BEST]))
common.save_proba(test_df["id"], test_proba, clf.classes_, f"tfidf_{BEST}", "test")
submission = common.save_submission(test_df["id"], clf.classes_[test_proba.argmax(1)], sample_sub, SUBMISSION_PATH)
print("best:", BEST, "| saved", SUBMISSION_PATH, submission.shape)

## Conclusions

| Finding | Decision |
|---|---|
| <!-- TODO --> | <!-- TODO --> |
| <!-- TODO --> | <!-- TODO --> |